# 02 · Set one temperature target

Apply one target after confirming the request. Readback verifies the reported target, not thermal settling. Inspect equipment after an uncertain write; never automatically replay it.

Run top to bottom in a **Python 3.12+** kernel. This notebook includes all configuration and setup; no sibling example files are imported. Run only one hardware notebook/process at a time.

**Prepare the filled cooling loop before connecting.** T255P connection sends U, takes serial control and can change AUTO START to RUN, even with target writes disabled. Software disconnect does not stop the pump/chiller or restore its target. Physical hardware has not been validated with this package.

## 1. Environment and imports

Select a kernel with this package and pyserial installed. From the repository root install with `%pip install -e .`; from the examples directory use `%pip install -e ..`. The script-only dashboard (04_dashboard.py) needs the `[gui]` extra, e.g. `%pip install -e "..[gui]"`. Restart the kernel after installation or driver source changes. Imports open no serial ports.

In [1]:
import sys
from pathlib import Path

# Prefer checkout sources when launched from the repository or examples folder.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src/tark_chiller/controller.py").is_file():
        sys.path.insert(0, str(candidate / "src"))
        break

In [2]:
from tark_chiller.controller import T255P, TarkChiller

## 2. Choose the port

This example connects to a **ThermoTek T255P**. Set `PORT` to its serial port (for example `"COM9"`). Adapter metadata identifies the USB adapter, not the attached chiller.

The connection cell below spells out the T255P profile: 9600 baud, 8N1, XON/XOFF, no hardware flow control, and RTS/DTR deasserted. Check that your adapter/cable supports these line levels. Distilled-water target bounds are 5–45 °C, in 0.1 °C steps. See the [hardware guide](../docs/hardware.md) for wiring and the separate CAL protocol.

In [3]:
PORT = "COM9"  # Set the identified serial port, e.g. "COM9".

TARGET_C = 20.0  # Choose an approved target within your setup limits.

## 3. Connection settings

The next operation cell constructs the driver directly with the settings shown. Construction opens nothing. Its context connects and releases the port, including on a read error or interrupt. Among the notebooks, only example 02 enables target writes; dashboard script 04 also enables explicit GUI target requests.

## 4. Set one temperature target

Apply one target after confirming the request. Readback verifies the reported target, not thermal settling. Inspect equipment after an uncertain write; never automatically replay it.

Run the next cell as a unit: it owns connection and cleanup together. Interrupt the cell to cancel; errors remain visible. Check the printed existing and requested targets before pressing Enter. Interrupt before confirming to cancel the target request.

In [4]:
if not isinstance(PORT, str) or not PORT.strip():
    raise ValueError('Hardware not configured: set PORT, e.g. "COM9". No port was opened.')

chiller = TarkChiller(
    {
        "port": PORT,
        "baudrate": 9600,
        "bytesize": 8,
        "parity": "N",
        "stopbits": 1,
        "xonxoff": True,
        "rtscts": False,
        "dsrdtr": False,
        "rts": False,
        "dtr": False,
    },
    protocol=T255P(allow_writes=True),
    setpoint_range=(5.0, 45.0),
    coolant_source="ThermoTek T255P Rev C pp9/20; verify installed distilled water",
)

with chiller:
    print(f"Existing setpoint: {chiller.read_setpoint():.2f} C")
    input(f"Press Enter to apply {TARGET_C:g} C; Ctrl+C to cancel. ")

    # 2. Send one request. Values are checked; uncertain writes are never retried.
    # T255P requires RUN with no alarms and suitable coolant/setup.
    chiller.set_setpoint(TARGET_C)

    # 3. Read the target again. This is a target check, not a settling test.
    reported_c = chiller.read_setpoint()
    print(f"Reported setpoint: {reported_c:.2f} C")
    if reported_c != TARGET_C:
        raise RuntimeError("Target differs from the request. Inspect the equipment; do not retry.")
    print("Readback matches. Observe the actual liquid temperature separately.")

# 4. The connection is closed; the equipment keeps its current target/state.
# No automatic restoration, pump control or equipment stop is sent.

Existing setpoint: 18.00 C
Reported setpoint: 20.00 C
Readback matches. Observe the actual liquid temperature separately.


## 5. Check results and finish

Observe the actual liquid temperature separately. Matching target readback does not establish settling.

Context cleanup closes software resources, including after a read error or kernel interrupt. If cleanup fails, resources may remain owned: allow pending I/O to finish and retry `chiller.disconnect(timeout_s=5.0)` if `chiller` was created. Do not rerun configuration over a live connection. A forced kernel termination cannot guarantee cleanup. Follow the equipment's physical shutdown procedure separately.